# Phase 4 — EDA: Load, Inspect & Join (DuckDB)

**Project:** Flight Delay Prediction  
**Stack:** DuckDB + Pandas (no Spark needed for EDA on a single machine)  

DuckDB reads CSV files directly off disk using streaming — it never loads the full dataset into RAM at once. On a 16GB machine this is dramatically faster than PySpark for local EDA.

### Sections
1. Setup & install
2. Connect DuckDB and register CSV folders
3. BTS inspection
4. Weather inspection
5. Aggregate weather hourly → daily
6. Join BTS + weather
7. EDA queries
8. Save joined dataset as Parquet

---
## 1. Install & Import

In [1]:
# Run once
# !pip install duckdb pandas pyarrow

In [2]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

# ── Paths ────────────────────────────────────────────────
PROJECT_ROOT    = Path("D:/project")
BTS_DIR         = PROJECT_ROOT / "data" / "raw" / "bts"
WEATHER_DIR     = PROJECT_ROOT / "data" / "raw" / "weather"
PROCESSED_DIR   = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BTS_GLOB     = str(BTS_DIR     / "bts_filtered_*.csv").replace("\\", "/")
WEATHER_GLOB = str(WEATHER_DIR / "weather_*.csv").replace("\\", "/")
PARQUET_OUT  = str(PROCESSED_DIR / "bts_weather_joined.parquet").replace("\\", "/")

print(f"BTS glob     : {BTS_GLOB}")
print(f"Weather glob : {WEATHER_GLOB}")
print(f"Output       : {PARQUET_OUT}")

BTS glob     : D:/project/data/raw/bts/bts_filtered_*.csv
Weather glob : D:/project/data/raw/weather/weather_*.csv
Output       : D:/project/data/processed/bts_weather_joined.parquet


---
## 2. Connect DuckDB

In [3]:
# Persistent DB file — stores intermediate views so you don't re-scan CSVs each run
DB_PATH = str(PROCESSED_DIR / "flight_delay.duckdb").replace("\\", "/")

con = duckdb.connect(DB_PATH)

# Use up to 8GB RAM and 4 threads — safe for 16GB machine
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")

print(f"DuckDB version : {duckdb.__version__}")
print(f"DB file        : {DB_PATH}")

DuckDB version : 1.5.1
DB file        : D:/project/data/processed/flight_delay.duckdb


---
## 3. BTS Data Inspection

In [4]:
# How many rows and columns total across all 120 CSVs?
con.execute(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        MIN(FlightDate)                   AS earliest_date,
        MAX(FlightDate)                   AS latest_date,
        COUNT(DISTINCT Origin)            AS unique_origins,
        COUNT(DISTINCT Reporting_Airline) AS unique_carriers
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
""").df()

,total_rows,earliest_date,latest_date,unique_origins,unique_carriers
0,48523795,2015-01-01,2024-12-31,352,20


In [5]:
# Schema — what columns and types did DuckDB infer?
con.execute(f"""
    DESCRIBE SELECT * FROM read_csv_auto('{BTS_GLOB}', union_by_name=true) LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,FlightDate,DATE,YES,None,None,None
1,Reporting_Airline,VARCHAR,YES,None,None,None
2,Flight_Number_Reporting_Airline,BIGINT,YES,None,None,None
3,Origin,VARCHAR,YES,None,None,None
4,Dest,VARCHAR,YES,None,None,None
5,CRSDepTime,VARCHAR,YES,None,None,None
6,DepTime,VARCHAR,YES,None,None,None
7,DepDelay,DOUBLE,YES,None,None,None
8,DepDelayMinutes,DOUBLE,YES,None,None,None
9,CRSArrTime,VARCHAR,YES,None,None,None


In [6]:
# Quick preview
con.execute(f"""
    SELECT * FROM read_csv_auto('{BTS_GLOB}', union_by_name=true) LIMIT 5
""").df()

,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2015-01-01,AA,1,JFK,LAX,0900,0855,-5.00,0.00,1230,1237,7.00,7.00,0.00,None,0.00,390.00,402.00,2475.00,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,AA,1,JFK,LAX,0900,0850,-10.00,0.00,1230,1211,-19.00,0.00,0.00,None,0.00,390.00,381.00,2475.00,NaN,NaN,NaN,NaN,NaN
2,2015-01-03,AA,1,JFK,LAX,0900,0853,-7.00,0.00,1230,1151,-39.00,0.00,0.00,None,0.00,390.00,358.00,2475.00,NaN,NaN,NaN,NaN,NaN
3,2015-01-04,AA,1,JFK,LAX,0900,0853,-7.00,0.00,1230,1218,-12.00,0.00,0.00,None,0.00,390.00,385.00,2475.00,NaN,NaN,NaN,NaN,NaN
4,2015-01-05,AA,1,JFK,LAX,0900,0853,-7.00,0.00,1230,1222,-8.00,0.00,0.00,None,0.00,390.00,389.00,2475.00,NaN,NaN,NaN,NaN,NaN


In [7]:
# Null audit — % missing per key column
con.execute(f"""
    SELECT
        COUNT(*) AS total,
        ROUND(100.0 * SUM(CASE WHEN FlightDate       IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_FlightDate,
        ROUND(100.0 * SUM(CASE WHEN Origin            IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_Origin,
        ROUND(100.0 * SUM(CASE WHEN DepDelayMinutes   IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_DepDelayMins,
        ROUND(100.0 * SUM(CASE WHEN Cancelled         IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_Cancelled,
        ROUND(100.0 * SUM(CASE WHEN WeatherDelay      IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_WeatherDelay,
        ROUND(100.0 * SUM(CASE WHEN CarrierDelay      IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_CarrierDelay
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
""").df()

,total,pct_null_FlightDate,pct_null_Origin,pct_null_DepDelayMins,pct_null_Cancelled,pct_null_WeatherDelay,pct_null_CarrierDelay
0,48523795,0.00,0.00,1.84,0.00,81.66,81.66


In [8]:
# Flights and delay rate per airport
con.execute(f"""
    SELECT
        Origin                                                                              AS airport,
        COUNT(*)                                                                            AS total_flights,
        SUM(CASE WHEN DepDelayMinutes >= 15 THEN 1 ELSE 0 END)                             AS delayed_flights,
        ROUND(100.0 * AVG(CASE WHEN DepDelayMinutes >= 15 THEN 1.0 ELSE 0.0 END), 1)      AS delay_rate_pct,
        ROUND(AVG(CAST(DepDelayMinutes AS FLOAT)), 1)                                       AS avg_delay_mins
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
    WHERE DepDelayMinutes IS NOT NULL
    GROUP BY Origin
    ORDER BY delay_rate_pct DESC
""").df()

,airport,total_flights,delayed_flights,delay_rate_pct,avg_delay_mins
0,MGW,19,10.00,52.60,87.20
1,SWF,225,94.00,41.80,29.40
2,STC,78,25.00,32.10,23.40
3,SCK,4257,1324.00,31.10,23.10
4,BQN,7850,2401.00,30.60,24.30
...,...,...,...,...,...
347,BTM,6661,434.00,6.50,8.60
348,PIH,8281,499.00,6.00,6.20
349,EKO,5462,315.00,5.80,9.70
350,GUM,11,0.00,0.00,0.80


In [9]:
# Delay rate by year — watch for 2020 COVID anomaly
con.execute(f"""
    SELECT
        YEAR(CAST(FlightDate AS DATE))                                                        AS year,
        COUNT(*)                                                                             AS total_flights,
        ROUND(100.0 * AVG(CASE WHEN DepDelayMinutes >= 15 THEN 1.0 ELSE 0.0 END), 1)       AS delay_rate_pct
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
    WHERE DepDelayMinutes IS NOT NULL
    GROUP BY year
    ORDER BY year
""").df()

,year,total_flights,delay_rate_pct
0,2015,4611869,18.60
1,2016,4399247,17.40
2,2017,4395056,18.30
3,2018,5389505,18.30
4,2019,5504681,18.90
5,2020,3367950,9.10
6,2021,4564386,17.00
7,2022,4973586,20.80
8,2023,5121818,20.60
9,2024,5304427,21.00


In [10]:
# Delay cause breakdown (among delayed flights only)
con.execute(f"""
    SELECT
        ROUND(AVG(CAST(CarrierDelay      AS FLOAT)), 2) AS avg_carrier_delay,
        ROUND(AVG(CAST(WeatherDelay      AS FLOAT)), 2) AS avg_weather_delay,
        ROUND(AVG(CAST(NASDelay         AS FLOAT)), 2) AS avg_nas_delay,
        ROUND(AVG(CAST(LateAircraftDelay AS FLOAT)), 2) AS avg_late_aircraft_delay,
        ROUND(AVG(CAST(SecurityDelay    AS FLOAT)), 2) AS avg_security_delay
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
    WHERE DepDelayMinutes >= 15
""").df()

,avg_carrier_delay,avg_weather_delay,avg_nas_delay,avg_late_aircraft_delay,avg_security_delay
0,28.80,4.57,11.97,32.39,0.13


---
## 4. Weather Data Inspection

In [11]:
# Row count and date range
con.execute(f"""
    SELECT
        COUNT(*)              AS total_rows,
        MIN(FL_DATE)          AS earliest,
        MAX(FL_DATE)          AS latest,
        COUNT(DISTINCT AIRPORT_CODE) AS airports
    FROM read_csv_auto('{WEATHER_GLOB}', union_by_name=true)
""").df()

RuntimeError: Query interrupted

In [ ]:
# Hourly observation count per airport — expect ~87,600 per airport (8,760/yr × 10yr)
con.execute(f"""
    SELECT
        AIRPORT_CODE,
        COUNT(*) AS hourly_rows
    FROM read_csv_auto('{WEATHER_GLOB}', union_by_name=true)
    GROUP BY AIRPORT_CODE
    ORDER BY hourly_rows ASC
""").df()

,AIRPORT_CODE,hourly_rows
0,PHX,110389
1,LAS,110643
2,SFO,118953
3,EWR,121154
4,IAH,123282
5,LAX,124072
6,PDX,125935
7,ATL,126820
8,SAN,127084
9,AUS,129033


In [ ]:
# Weather summary stats
con.execute(f"""
    SELECT
        ROUND(MIN(TEMP_C), 1)          AS min_temp,
        ROUND(MAX(TEMP_C), 1)          AS max_temp,
        ROUND(AVG(TEMP_C), 1)          AS avg_temp,
        ROUND(MIN(VISIBILITY_KM), 1)   AS min_vis_km,
        ROUND(AVG(VISIBILITY_KM), 1)   AS avg_vis_km,
        ROUND(MAX(WIND_SPEED_MS), 1)   AS max_wind_ms,
        ROUND(AVG(WIND_SPEED_MS), 1)   AS avg_wind_ms,
        ROUND(MAX(PRECIP_MM), 1)       AS max_precip_mm
    FROM read_csv_auto('{WEATHER_GLOB}', union_by_name=true)
""").df()

,min_temp,max_temp,avg_temp,min_vis_km,avg_vis_km,max_wind_ms,avg_wind_ms,max_precip_mm
0,-30.60,48.30,16.80,0.00,43.20,45.30,3.60,9.90


---
## 5. Aggregate Weather: Hourly → Daily

Save this as a persistent DuckDB view so we don't recompute it on every query.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW weather_daily AS
    SELECT
        AIRPORT_CODE,
        FL_DATE,
        ROUND(AVG(TEMP_C), 2)           AS avg_temp_c,
        ROUND(MIN(TEMP_C), 2)           AS min_temp_c,
        ROUND(MAX(TEMP_C), 2)           AS max_temp_c,
        ROUND(AVG(DEW_POINT_C), 2)      AS avg_dew_point_c,
        ROUND(AVG(VISIBILITY_KM), 2)    AS avg_visibility_km,
        ROUND(MIN(VISIBILITY_KM), 2)    AS min_visibility_km,
        ROUND(AVG(WIND_SPEED_MS), 2)    AS avg_wind_ms,
        ROUND(MAX(WIND_SPEED_MS), 2)    AS max_wind_ms,
        ROUND(SUM(PRECIP_MM), 2)        AS total_precip_mm,
        ROUND(MAX(PRECIP_MM), 2)        AS max_precip_mm,
        ROUND(AVG(PRESSURE_HPA), 2)     AS avg_pressure_hpa,
        ROUND(AVG(CEILING_M), 0)        AS avg_ceiling_m,
        ROUND(MIN(CEILING_M), 0)        AS min_ceiling_m,
        COUNT(*)                        AS hourly_obs_count
    FROM read_csv_auto('{WEATHER_GLOB}', union_by_name=true)
    GROUP BY AIRPORT_CODE, FL_DATE
""")

# Verify
con.execute("SELECT COUNT(*) AS daily_rows FROM weather_daily").df()

,daily_rows
0,73060


---
## 6. Join BTS + Weather

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW bts_clean AS
    SELECT
        FlightDate                                                                   AS FL_DATE,
        Origin                                                                       AS AIRPORT_CODE,
        Dest,
        Reporting_Airline,
        CAST(CRSDepTime        AS INTEGER)                                           AS CRSDepTime,
        CAST(DepDelay          AS DOUBLE)                                            AS DepDelay,
        CAST(DepDelayMinutes   AS DOUBLE)                                            AS DepDelayMinutes,
        CAST(CASE WHEN DepDelayMinutes >= 15 THEN 1 ELSE 0 END AS INTEGER)          AS DEP_DEL15,
        CAST(ArrDelay          AS DOUBLE)                                            AS ArrDelay,
        CAST(ArrDelayMinutes   AS DOUBLE)                                            AS ArrDelayMinutes,
        CAST(CASE WHEN ArrDelayMinutes >= 15 THEN 1 ELSE 0 END AS INTEGER)          AS ARR_DEL15,
        CAST(Cancelled         AS INTEGER)                                           AS Cancelled,
        CancellationCode,
        CAST(Distance          AS DOUBLE)                                            AS Distance,
        CAST(CarrierDelay      AS DOUBLE)                                            AS CarrierDelay,
        CAST(WeatherDelay      AS DOUBLE)                                            AS WeatherDelay,
        CAST(NASDelay          AS DOUBLE)                                            AS NASDelay,
        CAST(SecurityDelay     AS DOUBLE)                                            AS SecurityDelay,
        CAST(LateAircraftDelay AS DOUBLE)                                            AS LateAircraftDelay
    FROM read_csv_auto('{BTS_GLOB}', union_by_name=true)
    WHERE FlightDate IS NOT NULL AND Origin IS NOT NULL
""")

print("bts_clean view created. DEP_DEL15 and ARR_DEL15 derived from delay minutes.")

bts_clean view created. DEP_DEL15 and ARR_DEL15 derived from delay minutes.


In [ ]:
# Create the joined view
con.execute("""
    CREATE OR REPLACE VIEW joined AS
    SELECT
        b.*,
        w.avg_temp_c,
        w.min_temp_c,
        w.max_temp_c,
        w.avg_dew_point_c,
        w.avg_visibility_km,
        w.min_visibility_km,
        w.avg_wind_ms,
        w.max_wind_ms,
        w.total_precip_mm,
        w.max_precip_mm,
        w.avg_pressure_hpa,
        w.avg_ceiling_m,
        w.min_ceiling_m,
        w.hourly_obs_count
    FROM bts_clean b
    LEFT JOIN weather_daily w
        ON b.AIRPORT_CODE = w.AIRPORT_CODE
        AND b.FL_DATE = w.FL_DATE
""")

# Quick row count check
con.execute("SELECT COUNT(*) AS joined_rows FROM joined").df()

,joined_rows
0,48523795


In [ ]:
# Weather match rate — how many flights got weather data?
con.execute("""
    SELECT
        COUNT(*)                                                        AS total_flights,
        SUM(CASE WHEN avg_temp_c IS NOT NULL THEN 1 ELSE 0 END)        AS with_weather,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS match_pct
    FROM joined
""").df()

,total_flights,with_weather,match_pct
0,48523795,30700603.00,63.30


---
## 7. EDA Queries on Joined Data

In [ ]:
# Delay rate per airport (joined dataset)
con.execute("""
    SELECT
        AIRPORT_CODE,
        COUNT(*)                                          AS total_flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1)  AS delay_rate_pct,
        ROUND(AVG(DepDelayMinutes), 1)                    AS avg_delay_mins,
        ROUND(AVG(avg_temp_c), 1)                         AS avg_temp_c,
        ROUND(AVG(avg_wind_ms), 1)                        AS avg_wind_ms,
        ROUND(AVG(total_precip_mm), 2)                    AS avg_precip_mm
    FROM joined
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY AIRPORT_CODE
    ORDER BY delay_rate_pct DESC
""").df()

,AIRPORT_CODE,total_flights,delay_rate_pct,avg_delay_mins,avg_temp_c,avg_wind_ms,avg_precip_mm
0,MGW,19,52.60,87.20,NaN,NaN,NaN
1,SWF,229,41.00,29.40,NaN,NaN,NaN
2,STC,83,30.10,23.40,NaN,NaN,NaN
3,SCK,4418,30.00,23.10,NaN,NaN,NaN
4,BQN,8075,29.70,24.30,NaN,NaN,NaN
...,...,...,...,...,...,...,...
347,BTM,6739,6.40,8.60,NaN,NaN,NaN
348,PIH,8313,6.00,6.20,NaN,NaN,NaN
349,EKO,5530,5.70,9.70,NaN,NaN,NaN
350,EFD,1,0.00,7.00,NaN,NaN,NaN


In [ ]:
# Weather conditions: delayed vs on-time comparison
con.execute("""
    SELECT
        DEP_DEL15,
        COUNT(*)                              AS flights,
        ROUND(AVG(avg_temp_c), 2)             AS avg_temp_c,
        ROUND(AVG(avg_wind_ms), 2)            AS avg_wind_ms,
        ROUND(AVG(total_precip_mm), 2)        AS avg_precip_mm,
        ROUND(AVG(min_visibility_km), 2)      AS avg_min_visibility_km,
        ROUND(AVG(min_ceiling_m), 0)          AS avg_min_ceiling_m
    FROM joined
    WHERE DEP_DEL15 IS NOT NULL AND avg_temp_c IS NOT NULL
    GROUP BY DEP_DEL15
    ORDER BY DEP_DEL15
""").df()

,DEP_DEL15,flights,avg_temp_c,avg_wind_ms,avg_precip_mm,avg_min_visibility_km,avg_min_ceiling_m
0,0,24885948,16.70,3.63,5.59,11.87,1221.00
1,1,5814655,17.12,3.79,5.80,10.72,1052.00


In [ ]:
# Delay rate by month — seasonality check
con.execute("""
    SELECT
        MONTH(CAST(FL_DATE AS DATE))                     AS month,
        COUNT(*)                                         AS total_flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1) AS delay_rate_pct
    FROM joined
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY month
    ORDER BY month
""").df()

,month,total_flights,delay_rate_pct
0,1,3933012,17.60
1,2,3646142,17.00
2,3,4245838,16.80
3,4,3891970,16.70
4,5,3975678,18.80
5,6,4056548,22.80
6,7,4300315,22.80
7,8,4312365,20.30
8,9,3989592,14.30
9,10,4167351,14.70


In [ ]:
# Delay rate by year (COVID anomaly check)
con.execute("""
    SELECT
        YEAR(CAST(FL_DATE AS DATE))                      AS year,
        COUNT(*)                                         AS total_flights,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1) AS delay_rate_pct
    FROM joined
    WHERE DEP_DEL15 IS NOT NULL
    GROUP BY year
    ORDER BY year
""").df()

,year,total_flights,delay_rate_pct
0,2015,4680502,18.30
1,2016,4447929,17.20
2,2017,4456437,18.00
3,2018,5469982,18.10
4,2019,5601002,18.60
5,2020,3570635,8.60
6,2021,4641693,16.70
7,2022,5097112,20.30
8,2023,5184157,20.40
9,2024,5374346,20.70


---
## 8. Save Joined Dataset as Parquet

Parquet is the best format to hand off to Phase 5 — compressed, fast to read, column-oriented.  
PySpark (used later for modeling) reads Parquet natively.

In [12]:
import os, duckdb
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
DB_PATH_FIXED      = str(Path("D:/project/data/processed/flight_delay.duckdb").resolve())
PARQUET_OUT_FIXED  = str(Path("D:/project/data/processed/bts_weather_joined.parquet").resolve())
TEMP_DIR_FIXED     = str(Path("D:/project/data/processed/tmp").resolve())
BTS_GLOB_FIXED     = "D:/project/data/raw/bts/bts_filtered_*.csv"
WEATHER_GLOB_FIXED = "D:/project/data/raw/weather/weather_*.csv"

def sql_path(p: str) -> str:
    return p.replace("\\", "/")

os.makedirs(Path(DB_PATH_FIXED).parent, exist_ok=True)
os.makedirs(TEMP_DIR_FIXED, exist_ok=True)

# ── Connection ─────────────────────────────────────────────────────────────────
try:
    con.close()
except:
    pass

con = duckdb.connect(DB_PATH_FIXED)
os.makedirs(TEMP_DIR_FIXED, exist_ok=True)                              # ← add
con.execute(f"SET temp_directory='{sql_path(TEMP_DIR_FIXED)}'")        # ← add
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")

# ── Views ──────────────────────────────────────────────────────────────────────
con.execute(f"""
    CREATE OR REPLACE VIEW weather_daily AS
    SELECT AIRPORT_CODE, FL_DATE,
        ROUND(AVG(TEMP_C), 2)         AS avg_temp_c,
        ROUND(MIN(TEMP_C), 2)         AS min_temp_c,
        ROUND(MAX(TEMP_C), 2)         AS max_temp_c,
        ROUND(AVG(DEW_POINT_C), 2)    AS avg_dew_point_c,
        ROUND(AVG(VISIBILITY_KM), 2)  AS avg_visibility_km,
        ROUND(MIN(VISIBILITY_KM), 2)  AS min_visibility_km,
        ROUND(AVG(WIND_SPEED_MS), 2)  AS avg_wind_ms,
        ROUND(MAX(WIND_SPEED_MS), 2)  AS max_wind_ms,
        ROUND(SUM(PRECIP_MM), 2)      AS total_precip_mm,
        ROUND(MAX(PRECIP_MM), 2)      AS max_precip_mm,
        ROUND(AVG(PRESSURE_HPA), 2)   AS avg_pressure_hpa,
        ROUND(AVG(CEILING_M), 0)      AS avg_ceiling_m,
        ROUND(MIN(CEILING_M), 0)      AS min_ceiling_m,
        COUNT(*)                       AS hourly_obs_count
    FROM read_csv_auto('{sql_path(WEATHER_GLOB_FIXED)}', union_by_name=true)
    GROUP BY AIRPORT_CODE, FL_DATE
""")

con.execute(f"""
    CREATE OR REPLACE VIEW bts_clean AS
    SELECT
        FlightDate                                                          AS FL_DATE,
        Origin                                                              AS AIRPORT_CODE,
        Dest, Reporting_Airline,
        CAST(CRSDepTime        AS INTEGER)                                 AS CRSDepTime,
        CAST(DepDelay          AS DOUBLE)                                  AS DepDelay,
        CAST(DepDelayMinutes   AS DOUBLE)                                  AS DepDelayMinutes,
        CAST(CASE WHEN DepDelayMinutes >= 15 THEN 1 ELSE 0 END AS INTEGER) AS DEP_DEL15,
        CAST(ArrDelay          AS DOUBLE)                                  AS ArrDelay,
        CAST(ArrDelayMinutes   AS DOUBLE)                                  AS ArrDelayMinutes,
        CAST(CASE WHEN ArrDelayMinutes >= 15 THEN 1 ELSE 0 END AS INTEGER) AS ARR_DEL15,
        CAST(Cancelled         AS INTEGER)                                 AS Cancelled,
        CancellationCode,
        CAST(Distance          AS DOUBLE)                                  AS Distance,
        CAST(CarrierDelay      AS DOUBLE)                                  AS CarrierDelay,
        CAST(WeatherDelay      AS DOUBLE)                                  AS WeatherDelay,
        CAST(NASDelay          AS DOUBLE)                                  AS NASDelay,
        CAST(SecurityDelay     AS DOUBLE)                                  AS SecurityDelay,
        CAST(LateAircraftDelay AS DOUBLE)                                  AS LateAircraftDelay
    FROM read_csv_auto('{sql_path(BTS_GLOB_FIXED)}', union_by_name=true)
    WHERE FlightDate IS NOT NULL AND Origin IS NOT NULL
""")

con.execute("""
    CREATE OR REPLACE VIEW joined AS
    SELECT b.*,
        w.avg_temp_c, w.min_temp_c, w.max_temp_c, w.avg_dew_point_c,
        w.avg_visibility_km, w.min_visibility_km, w.avg_wind_ms, w.max_wind_ms,
        w.total_precip_mm, w.max_precip_mm, w.avg_pressure_hpa,
        w.avg_ceiling_m, w.min_ceiling_m, w.hourly_obs_count
    FROM bts_clean b
    LEFT JOIN weather_daily w
           ON b.AIRPORT_CODE = w.AIRPORT_CODE
          AND b.FL_DATE      = w.FL_DATE
""")

# ── Write Parquet ──────────────────────────────────────────────────────────────
print("Views ready. Writing Parquet — this may take a few minutes...")

con.execute(f"""
    COPY (SELECT * FROM joined)
    TO '{sql_path(PARQUET_OUT_FIXED)}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Done. Saved to: {PARQUET_OUT_FIXED}")

Views ready. Writing Parquet — this may take a few minutes...
Done. Saved to: D:\project\data\processed\bts_weather_joined.parquet


In [13]:
# Verify the saved Parquet reads back correctly
verify = con.execute(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT AIRPORT_CODE) AS airports
    FROM read_parquet('{PARQUET_OUT}')
""").df()

print(verify)
print(f"\nParquet file size: {Path(PARQUET_OUT).stat().st_size / 1e6:.1f} MB")

       rows  airports
0  48523795       352

Parquet file size: 485.6 MB


---
## 9. Summary

In [14]:
summary = con.execute("""
    SELECT
        COUNT(*)                                          AS total_flights,
        COUNT(DISTINCT AIRPORT_CODE)                      AS airports,
        MIN(FL_DATE)                                      AS date_from,
        MAX(FL_DATE)                                      AS date_to,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1)  AS overall_delay_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS weather_match_pct
    FROM joined
    WHERE DEP_DEL15 IS NOT NULL
""").df()

print("=" * 55)
print("  PHASE 4 COMPLETE")
print("=" * 55)
for col in summary.columns:
    print(f"  {col:<30} : {summary[col].iloc[0]}")
print(f"  {'Parquet saved to':<30} : {PARQUET_OUT}")
print("=" * 55)
print("\n  Next → Phase 5: Feature Engineering")
print("  - Weather severity flags (low vis, high wind, heavy rain)")
print("  - Time features: hour-of-day, day-of-week, month, season")
print("  - Hub vs non-hub flag")
print("  - Route frequency / congestion proxy")

con.close()

  PHASE 4 COMPLETE
  total_flights                  : 48523795
  airports                       : 352
  date_from                      : 2015-01-01 00:00:00
  date_to                        : 2024-12-31 00:00:00
  overall_delay_rate_pct         : 18.0
  weather_match_pct              : 63.3
  Parquet saved to               : D:/project/data/processed/bts_weather_joined.parquet

  Next → Phase 5: Feature Engineering
  - Weather severity flags (low vis, high wind, heavy rain)
  - Time features: hour-of-day, day-of-week, month, season
  - Hub vs non-hub flag
  - Route frequency / congestion proxy


In [1]:
import duckdb
con = duckdb.connect()
result = con.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) AS missing_weather,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
    FROM read_parquet('D:/project/data/processed/features_final.parquet')
""").df()
con.close()
print(result)

      total  missing_weather  pct_missing
0  48523795       17823192.0         36.7


In [2]:
import duckdb
con = duckdb.connect()
con.execute("""
    SELECT
        AIRPORT_CODE,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) AS missing_weather,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
    FROM read_parquet('D:/project/data/processed/features_final.parquet')
    GROUP BY AIRPORT_CODE
    ORDER BY pct_missing DESC
""").df()

,AIRPORT_CODE,total_flights,missing_weather,pct_missing
0,MLB,21120,21120.0,100.0
1,LCK,54,54.0,100.0
2,RIC,126630,126630.0,100.0
3,LIT,87474,87474.0,100.0
4,PSP,107273,107273.0,100.0
...,...,...,...,...
347,LAS,1582919,0.0,0.0
348,AUS,642525,0.0,0.0
349,ORD,2726718,0.0,0.0
350,LAX,1958952,0.0,0.0


In [3]:
import duckdb
con = duckdb.connect()
con.execute("""
    SELECT
        AIRPORT_CODE,
        COUNT(*) AS total_flights,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
    FROM read_parquet('D:/project/data/processed/features_final.parquet')
    WHERE AIRPORT_CODE NOT IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    )
    GROUP BY AIRPORT_CODE
    ORDER BY total_flights DESC
    LIMIT 10
""").df()

,AIRPORT_CODE,total_flights,pct_missing
0,MCO,585508,100.0
1,MSP,513990,100.0
2,LGA,509261,100.0
3,DTW,483192,100.0
4,DCA,435792,100.0
5,PHL,432371,100.0
6,FLL,403643,100.0
7,BWI,353547,100.0
8,BNA,349482,100.0
9,SMF,336705,100.0


In [1]:
import duckdb
con = duckdb.connect()
con.execute("""
    SELECT
        AIRPORT_CODE,
        COUNT(*) AS total,
        SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) AS missing,
        ROUND(100.0 * SUM(CASE WHEN avg_temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing,
        MIN(FL_DATE) AS earliest,
        MAX(FL_DATE) AS latest
    FROM read_parquet('D:/project/data/processed/features_final.parquet')
    WHERE AIRPORT_CODE IN (
        'ATL','LAX','ORD','DFW','DEN','JFK','SFO','CLT','LAS','PHX',
        'MIA','IAH','SEA','EWR','BOS','SLC','SAN','TPA','PDX','AUS'
    )
    GROUP BY AIRPORT_CODE
    ORDER BY pct_missing DESC
""").df()

,AIRPORT_CODE,total,missing,pct_missing,earliest,latest
0,DFW,2603267,6093.0,0.2,2015-01-01,2024-12-31
1,LAS,1582919,0.0,0.0,2015-01-01,2024-12-31
2,PHX,1641017,0.0,0.0,2015-01-01,2024-12-31
3,CLT,1779704,0.0,0.0,2015-01-01,2024-12-31
4,SEA,1444676,0.0,0.0,2015-01-01,2024-12-31
5,JFK,1083844,0.0,0.0,2015-01-01,2024-12-31
6,ATL,3478431,0.0,0.0,2015-01-01,2024-12-31
7,LAX,1958952,0.0,0.0,2015-01-01,2024-12-31
8,ORD,2726718,0.0,0.0,2015-01-01,2024-12-31
9,AUS,642525,0.0,0.0,2015-01-01,2024-12-31
